# Download project dependencies and datasets

The primary dataset is now **OSMa-Bench**, downloaded as one official `data.zip` archive. Run the OSMa-Bench cell below on its own, or use Run All to also check/download the ConceptGraphs and OSMa-Bench source repositories. The previous OpenEQA download is retained as an optional section (disabled by default).

## Download ConceptGraphs source

Clone the [official repository](https://github.com/concept-graphs/concept-graphs) into `dependencies/concept-graphs` for code analysis. Requires Git and network access. Run from the project root or the `code` folder. This downloads source only; model weights, datasets, and runtime packages are separate setup steps.

Before downloading, list the contents of `dependencies`. Reuse an existing ConceptGraphs checkout only after checking its origin, commit, Git object integrity, and every tracked file. Existing files are preserved; incomplete checkouts are reported instead of overwritten. Other dependency folders are listed, not assumed to be valid installations. The printed commit identifies the version being studied.

In [1]:
from pathlib import Path
import shutil
import subprocess
import tempfile
import os
import re
import time
import threading
import queue
import html
from IPython.display import HTML, display

def clone_with_progress(url, destination, idle_timeout=120, total_timeout=600):
    # Git reports separate percentages for receiving, resolving and checkout.
    panel = display(HTML('<progress></progress> Connecting to GitHub...'), display_id=True)
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    command = ['git', '-c', 'http.lowSpeedLimit=1024', '-c', 'http.lowSpeedTime=60', 'clone', '--depth', '1', '--single-branch', '--progress', url, str(destination)]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', env=env)
    messages = queue.Queue()
    def read_output():
        try:
            for line in process.stdout:
                messages.put(line.strip())
        finally:
            messages.put(None)
    reader = threading.Thread(target=read_output, daemon=True)
    reader.start()
    start = last_activity = time.monotonic()
    recent = []
    status = 'Connecting to GitHub...'
    percent = None
    try:
        while True:
            try:
                line = messages.get(timeout=0.5)
            except queue.Empty:
                line = ''
            if line is None:
                break
            now = time.monotonic()
            if line:
                last_activity = now
                recent = (recent + [line])[-15:]
                status = line
                match = re.search(r'(\d+)%', line)
                percent = int(match.group(1)) if match else None
            value = '' if percent is None else f' value={percent} max=100'
            panel.update(HTML(f'<progress{value}></progress> {html.escape(status)}<br>Elapsed: {now-start:.0f}s; no new output: {now-last_activity:.0f}s'))
            if now - last_activity > idle_timeout or now - start > total_timeout:
                raise TimeoutError('Download stalled or exceeded 10 minutes. Check GitHub connectivity before retrying.')
        if process.wait(timeout=10):
            raise RuntimeError('Git download failed:' + chr(10) + chr(10).join(recent))
        panel.update(HTML('<progress value=100 max=100></progress> Download complete; verifying files...'))
    except BaseException:
        if process.poll() is None:
            if os.name == 'nt':
                subprocess.run(['taskkill', '/PID', str(process.pid), '/T', '/F'], capture_output=True)
            else:
                process.kill()
            process.wait(timeout=10)
        panel.update(HTML('<b>Download stopped.</b> See the error below.'))
        raise
    finally:
        reader.join(timeout=2)
        process.stdout.close()

def run_git(*args):
    result = subprocess.run(['git', *map(str, args)], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f'Git failed (exit {result.returncode}): {result.stderr.strip() or result.stdout.strip()}')
    return result.stdout.strip()

def verify_checkout(path):
    run_git('-C', path, 'rev-parse', '--verify', 'HEAD^{commit}')
    run_git('-C', path, 'fsck', '--connectivity-only', '--no-dangling')
    tracked = run_git('-C', path, 'ls-tree', '-r', '--name-only', '-z', 'HEAD').split(chr(0))
    missing = [name for name in tracked if name and not (path / name).exists()]
    if missing:
        raise RuntimeError(f'{len(missing)} tracked files are missing: {missing[:10]}')
    changes = run_git('-C', path, 'status', '--porcelain')
    if changes:
        print('Local changes detected and preserved:' + chr(10) + changes)
    print(f'Verified {sum(bool(name) for name in tracked)} tracked files.')

if shutil.which('git') is None:
    raise RuntimeError('Install Git and restart the notebook kernel first.')

cwd = Path.cwd().resolve()
project_root = next((p for p in (cwd, *cwd.parents) if (p / 'code' / 'download_dependencies.ipynb').is_file()), None)
if project_root is None:
    raise RuntimeError('Run this notebook from the capstone project folder or its code folder.')
repo_url = 'https://github.com/concept-graphs/concept-graphs.git'
repo_path = project_root / 'dependencies' / 'concept-graphs'
repo_path.parent.mkdir(parents=True, exist_ok=True)
print(f'Dependencies folder: {repo_path.parent}')
entries = sorted(repo_path.parent.iterdir(), key=lambda p: p.name.lower())
if not entries:
    print('No dependencies downloaded yet.')
for entry in entries:
    kind = 'folder' if entry.is_dir() else 'file'
    note = ' (ConceptGraphs: will verify)' if entry == repo_path else ' (unmanaged: preserved, not validated)'
    print(f'  [{kind}] {entry.name}{note}')

if not repo_path.exists():
    # Publish the final folder only after a successful, verified clone.
    staging = Path(tempfile.mkdtemp(prefix='concept-graphs-download-', dir=repo_path.parent))
    try:
        staged_repo = Path(staging) / 'repo'
        print('Downloading ConceptGraphs; wait for the completion message.')
        clone_with_progress(repo_url, staged_repo)
        verify_checkout(staged_repo)
        staged_repo.rename(repo_path)
        staging.rmdir()
    except BaseException:
        print(f'Incomplete download preserved at {staging}; the next run uses a fresh staging folder.')
        raise
elif not (repo_path / '.git').exists():
    raise RuntimeError(f'{repo_path} exists but is not a Git checkout. Inspect it before retrying.')
else:
    origin = run_git('-C', repo_path, 'remote', 'get-url', 'origin')
    if origin != repo_url:
        raise RuntimeError(f'Unexpected repository origin: {origin}')
    try:
        verify_checkout(repo_path)
    except RuntimeError as exc:
        raise RuntimeError(f'Checkout is incomplete at {repo_path}. Wait for any active download to finish. If it has failed, rename this folder to preserve it, then rerun this cell. Details: {exc}') from exc
    print('Using existing checkout; no files changed.')

print(f'Source: {repo_path}')
for args in (['branch', '--show-current'], ['rev-parse', 'HEAD']):
    print(run_git('-C', repo_path, *args))
print('ConceptGraphs source is ready.')


Dependencies folder: D:\School Stuff\shanghai work\Senior Capstone\dependencies
  [folder] concept-graphs (ConceptGraphs: will verify)
  [folder] open-eqa (unmanaged: preserved, not validated)
Verified 70 tracked files.
Using existing checkout; no files changed.
Source: D:\School Stuff\shanghai work\Senior Capstone\dependencies\concept-graphs
main
93277a02bd89171f8121e84203121cf7af9ebb5d
ConceptGraphs source is ready.


## Download OSMa-Bench source repository

Clone the [official OSMa-Bench repository](https://github.com/be2rlab/OSMa-Bench) into `dependencies/osma-bench`. This contains the benchmark source, including the semantic-segmentation and VQA workflows. The pre-generated dataset is downloaded separately by the dataset cell below.

Run this **self-contained cell** from the project root or `code` folder. It requires Git and network access for the first clone. It uses the default branch and prints the exact checked-out commit so you can record the version used for your project.

- Existing checkouts are reused without downloading or pulling updates after checking the origin, commit, Git object integrity, and presence of tracked files. Local modifications are reported and preserved.
- New clones are staged under `dependencies/` and moved to `dependencies/osma-bench` only after verification. Failed staging folders are preserved for inspection; rerunning starts a fresh staging folder.
- An unexpected origin, a non-Git destination, or an incomplete checkout stops with an explanation instead of overwriting files.
- This downloads the top-level source repository only; it does not install runtime packages, fetch model weights, or recursively initialize submodules. Follow the repository's workflow-specific README when you are ready to run its benchmark.

The existing `/dependencies/` Git ignore rule covers this checkout.


In [2]:
from pathlib import Path
import shutil
import subprocess
import tempfile
import os
import re
import time
import threading
import queue
import html
from IPython.display import HTML, display

def osma_clone_with_progress(url, destination, idle_timeout=120, total_timeout=600):
    # Git reports separate percentages for receiving, resolving and checkout.
    panel = display(HTML('<progress></progress> Connecting to GitHub...'), display_id=True)
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    command = ['git', '-c', 'http.lowSpeedLimit=1024', '-c', 'http.lowSpeedTime=60', 'clone', '--depth', '1', '--single-branch', '--progress', url, str(destination)]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', env=env)
    messages = queue.Queue()
    def read_output():
        try:
            for line in process.stdout:
                messages.put(line.strip())
        finally:
            messages.put(None)
    reader = threading.Thread(target=read_output, daemon=True)
    reader.start()
    start = last_activity = time.monotonic()
    recent = []
    status = 'Connecting to GitHub...'
    percent = None
    try:
        while True:
            try:
                line = messages.get(timeout=0.5)
            except queue.Empty:
                line = ''
            if line is None:
                break
            now = time.monotonic()
            if line:
                last_activity = now
                recent = (recent + [line])[-15:]
                status = line
                match = re.search(r'(\d+)%', line)
                percent = int(match.group(1)) if match else None
            value = '' if percent is None else f' value={percent} max=100'
            panel.update(HTML(f'<progress{value}></progress> {html.escape(status)}<br>Elapsed: {now-start:.0f}s; no new output: {now-last_activity:.0f}s'))
            if now - last_activity > idle_timeout or now - start > total_timeout:
                raise TimeoutError('Download stalled or exceeded 10 minutes. Check GitHub connectivity before retrying.')
        if process.wait(timeout=10):
            raise RuntimeError('Git download failed:' + chr(10) + chr(10).join(recent))
        panel.update(HTML('<progress value=100 max=100></progress> Download complete; verifying files...'))
    except BaseException:
        if process.poll() is None:
            if os.name == 'nt':
                subprocess.run(['taskkill', '/PID', str(process.pid), '/T', '/F'], capture_output=True)
            else:
                process.kill()
            process.wait(timeout=10)
        panel.update(HTML('<b>Download stopped.</b> See the error below.'))
        raise
    finally:
        reader.join(timeout=2)
        process.stdout.close()

def osma_run_git(*args):
    result = subprocess.run(['git', *map(str, args)], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f'Git failed (exit {result.returncode}): {result.stderr.strip() or result.stdout.strip()}')
    return result.stdout.strip()

def osma_verify_checkout(path):
    osma_run_git('-C', path, 'rev-parse', '--verify', 'HEAD^{commit}')
    osma_run_git('-C', path, 'fsck', '--connectivity-only', '--no-dangling')
    tracked = osma_run_git('-C', path, 'ls-tree', '-r', '--name-only', '-z', 'HEAD').split(chr(0))
    missing = [name for name in tracked if name and not (path / name).exists()]
    if missing:
        raise RuntimeError(f'{len(missing)} tracked files are missing: {missing[:10]}')
    changes = osma_run_git('-C', path, 'status', '--porcelain')
    if changes:
        print('Local changes detected and preserved:' + chr(10) + changes)
    print(f'Verified {sum(bool(name) for name in tracked)} tracked files.')

if shutil.which('git') is None:
    raise RuntimeError('Install Git and restart the notebook kernel first.')

cwd = Path.cwd().resolve()
project_root = next((p for p in (cwd, *cwd.parents) if (p / 'code' / 'download_dependencies.ipynb').is_file()), None)
if project_root is None:
    raise RuntimeError('Run this notebook from the capstone project folder or its code folder.')
repo_url = 'https://github.com/be2rlab/OSMa-Bench.git'
repo_path = project_root / 'dependencies' / 'osma-bench'
repo_path.parent.mkdir(parents=True, exist_ok=True)
print(f'Dependencies folder: {repo_path.parent}')
entries = sorted(repo_path.parent.iterdir(), key=lambda p: p.name.lower())
if not entries:
    print('No dependencies downloaded yet.')
for entry in entries:
    kind = 'folder' if entry.is_dir() else 'file'
    note = ' (OSMa-Bench: will verify)' if entry == repo_path else ' (unmanaged: preserved, not validated)'
    print(f'  [{kind}] {entry.name}{note}')

if not repo_path.exists():
    # Publish the final folder only after a successful, verified clone.
    staging = Path(tempfile.mkdtemp(prefix='osma-bench-download-', dir=repo_path.parent))
    try:
        staged_repo = Path(staging) / 'repo'
        print('Downloading OSMa-Bench; wait for the completion message.')
        osma_clone_with_progress(repo_url, staged_repo)
        osma_verify_checkout(staged_repo)
        staged_repo.rename(repo_path)
        staging.rmdir()
    except BaseException:
        print(f'Incomplete download preserved at {staging}; the next run uses a fresh staging folder.')
        raise
elif not (repo_path / '.git').exists():
    raise RuntimeError(f'{repo_path} exists but is not a Git checkout. Inspect it before retrying.')
else:
    origin = osma_run_git('-C', repo_path, 'remote', 'get-url', 'origin')
    if origin != repo_url:
        raise RuntimeError(f'Unexpected repository origin: {origin}')
    try:
        osma_verify_checkout(repo_path)
    except RuntimeError as exc:
        raise RuntimeError(f'Checkout is incomplete at {repo_path}. Wait for any active download to finish. If it has failed, rename this folder to preserve it, then rerun this cell. Details: {exc}') from exc
    print('Using existing checkout; no files changed.')

print(f'Source: {repo_path}')
for args in (['branch', '--show-current'], ['rev-parse', 'HEAD']):
    print(osma_run_git('-C', repo_path, *args))
print('OSMa-Bench source is ready.')


Dependencies folder: D:\School Stuff\shanghai work\Senior Capstone\dependencies
  [folder] concept-graphs (unmanaged: preserved, not validated)
  [folder] open-eqa (unmanaged: preserved, not validated)


Verified 1896 tracked files.
Source: D:\School Stuff\shanghai work\Senior Capstone\dependencies\osma-bench
master
7d9f33c756adaea2030711d24b08ca96d74ff344
OSMa-Bench source is ready.


## Download the OpenEQA HM3D and ScanNet scene histories

Download the per-episode archives from the `AIGeeksGroup/OpenEQA` Hugging Face mirror. Extracted scenes go to `dataset rgb only/frames/hm3d-v0` and `dataset rgb only/frames/scannet-v0`. Downloads are cached under `dataset rgb only/_downloads`. Existing non-empty episode folders are skipped, so the loop can be rerun after an interruption.

Install the downloader once if necessary with `python -m pip install huggingface_hub`.

**Optional legacy dataset:** set `DOWNLOAD_OPENEQA = True` in the next cell to run this existing download. It is disabled by default because OSMa-Bench is the primary dataset. Previously downloaded OpenEQA files are preserved.


In [ ]:
DOWNLOAD_OPENEQA = False

if DOWNLOAD_OPENEQA:
    import tarfile
    from pathlib import Path

    try:
        from huggingface_hub import HfApi, hf_hub_download
    except ImportError as exc:
        raise RuntimeError(
            'Install the downloader first: python -m pip install huggingface_hub'
        ) from exc

    RGB_REPO_ID = 'AIGeeksGroup/OpenEQA'
    RGB_SPLITS = ('hm3d-v0', 'scannet-v0')
    cwd = Path.cwd().resolve()
    project_root = next(
        (path for path in (cwd, *cwd.parents)
         if (path / 'code' / 'download_dependencies.ipynb').is_file()),
        None,
    )
    if project_root is None:
        raise RuntimeError(
            'Run this notebook from the capstone project folder or its code folder.'
        )
    RGB_ROOT = project_root / 'dataset rgb only'
    RGB_FRAMES_ROOT = RGB_ROOT / 'frames'
    RGB_DOWNLOAD_ROOT = RGB_ROOT / '_downloads'

    repo_files = HfApi().list_repo_files(RGB_REPO_ID, repo_type='dataset')

    for split in RGB_SPLITS:
        archives = sorted(
            name for name in repo_files
            if name.startswith(f'{split}/') and name.endswith('.tar')
        )
        if not archives:
            raise RuntimeError(f'No .tar episodes found for {split} in {RGB_REPO_ID}.')

        split_frames = RGB_FRAMES_ROOT / split
        split_frames.mkdir(parents=True, exist_ok=True)
        print(f'\n{split}: {len(archives)} episodes')

        for number, repo_file in enumerate(archives, 1):
            episode = Path(repo_file).stem
            destination = split_frames / episode
            if destination.is_dir() and any(destination.iterdir()):
                print(f'[{number}/{len(archives)}] {episode}: already extracted')
                continue

            print(f'[{number}/{len(archives)}] {episode}: downloading')
            archive = Path(hf_hub_download(
                repo_id=RGB_REPO_ID,
                repo_type='dataset',
                filename=repo_file,
                local_dir=RGB_DOWNLOAD_ROOT,
            ))

            staging = split_frames / f'.extracting-{episode}'
            if staging.exists():
                raise RuntimeError(
                    f'Incomplete extraction exists at {staging}. Inspect or remove it, then rerun.'
                )
            staging.mkdir()
            try:
                with tarfile.open(archive, 'r') as bundle:
                    bundle.extractall(staging, filter='data')
                extracted = staging / episode
                if not extracted.is_dir() or not any(extracted.iterdir()):
                    raise RuntimeError(f'{archive.name} did not contain {episode}/.')
                if destination.exists():
                    raise RuntimeError(f'Empty destination already exists: {destination}')
                extracted.rename(destination)
                staging.rmdir()
            except BaseException:
                print(f'Extraction stopped; temporary files remain at {staging}.')
                raise

            print(f'[{number}/{len(archives)}] {episode}: ready')

    print('\nHM3D and ScanNet scene-history download loop finished.')

else:
    print("OpenEQA download skipped (optional legacy dataset).")


## Download OSMa-Bench (primary dataset, one archive)

Run the following **single, self-contained cell** from the project root or `code` folder. It uses only Python's standard library; no Git Xet, Hugging Face package, HM3D asset download, or Habitat-Sim setup is needed to obtain these pre-generated observations.

- Official source: [warmhammer/OSMa-Bench_dataset, compressed branch](https://huggingface.co/datasets/warmhammer/OSMa-Bench_dataset/tree/compressed). The dataset card describes RGB, depth, semantic masks, camera trajectories, and validated QA. The archive's directory layout is preserved.
- Cached archive: `dataset/_downloads/osma-bench/data.zip` (3,218,010,116 bytes, about 3.22 GB / 3.00 GiB). The official SHA-256 and size are pinned below, verified on 2026-10-31. If upstream replaces the archive, this cell stops instead of silently accepting different data.
- Extracted dataset: `dataset/osma-bench/`. The completion message prints component counts and the path. Provenance and verification results are saved in `dataset/_downloads/osma-bench/verified.json`. These paths are covered by the existing `dataset/` Git ignore rule.
- Reruns verify and reuse the local archive **without network access**, then check every extracted file's size and ZIP CRC. A non-empty directory alone is never treated as a completed download.
- Interrupted downloads retain `data.zip.part` and resume with HTTP byte ranges (restart if the server does not support ranges). Interrupted extraction resumes in `dataset/.osma-bench.extracting/`, reusing only verified files; the final folder appears after successful extraction.
- Checks include SHA-256, ZIP CRC, available disk space, unsafe archive paths, and a lock against simultaneous runs. Keep enough free space for the archive **and** its uncompressed contents; the cell checks both stages.
- Existing damaged or modified files are preserved and reported. Move the reported file/folder aside before retrying; nothing is deleted automatically. Keep the cached ZIP for offline verification and recovery. Verification on reruns reads several GB and may take a little time.

The existing ConceptGraphs and optional OpenEQA sections above remain available. You can run this OSMa-Bench cell independently of them.


In [ ]:
from contextlib import contextmanager
import hashlib
from http.client import IncompleteRead
import json
import os
from pathlib import Path, PurePosixPath
import re
import shutil
import stat
import time
from urllib.error import URLError
from urllib.request import Request, urlopen
import zipfile
import zlib


REPO_ID = "warmhammer/OSMa-Bench_dataset"
REVISION = "compressed"
URL = f"https://huggingface.co/datasets/{REPO_ID}/resolve/{REVISION}/data.zip"
# Official Hugging Face LFS metadata, verified 2026-10-31. Fail closed if it changes.
ARCHIVE_SIZE = 3218010116
ARCHIVE_SHA256 = "e03b11abd0c9f6062296e5b348cd3c0049c3d5a18f3493fd5a3ed2d4e24f71fa"
CHUNK = 4 * 1024 * 1024


def require_space(folder, required):
    free = shutil.disk_usage(folder).free
    if free < required + 256 * 1024 * 1024:
        raise RuntimeError(f"Insufficient disk space at {folder}: need {required / 2**30:.2f} GiB plus 256 MiB reserve.")


@contextmanager
def download_lock(path):
    """OS-owned lock is automatically released even if the process is interrupted."""
    with path.open("a+b") as handle:
        if handle.tell() == 0:
            handle.write(b"0")
            handle.flush()
        handle.seek(0)
        try:
            if os.name == "nt":
                import msvcrt
                msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
            else:
                import fcntl
                fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except OSError as exc:
            raise RuntimeError("Another OSMa-Bench download is already running. Wait for it to finish.") from exc
        try:
            yield
        finally:
            handle.seek(0)
            if os.name == "nt":
                msvcrt.locking(handle.fileno(), msvcrt.LK_UNLCK, 1)
            else:
                fcntl.flock(handle, fcntl.LOCK_UN)


def verify_archive(path, expected_size, expected_sha256):
    if path.stat().st_size != expected_size:
        raise RuntimeError(f"Wrong archive size: {path}. Existing file preserved; move it aside before retrying.")
    print(f"Verifying SHA-256: {path.name}", flush=True)
    with path.open("rb") as handle:
        actual = hashlib.file_digest(handle, "sha256").hexdigest()
    if actual != expected_sha256:
        raise RuntimeError(f"SHA-256 mismatch: {path}. File preserved; move it aside before retrying.")


def download_archive(url, archive, expected_size, expected_sha256):
    if archive.exists():
        verify_archive(archive, expected_size, expected_sha256)
        print("Verified existing archive; no download needed.", flush=True)
        return
    partial = archive.with_suffix(archive.suffix + ".part")
    for attempt in range(1, 6):
        offset = partial.stat().st_size if partial.exists() else 0
        if offset > expected_size:
            raise RuntimeError(f"Oversized partial archive: {partial}. Move it aside before retrying.")
        if offset == expected_size:
            break
        require_space(archive.parent, expected_size - offset)
        headers = {"User-Agent": "capstone-osma-downloader/1.0", "Accept-Encoding": "identity"}
        if offset:
            headers["Range"] = f"bytes={offset}-"
        print(f"Download attempt {attempt}/5; starting at {offset / 2**20:.1f} / {expected_size / 2**20:.1f} MiB", flush=True)
        try:
            with urlopen(Request(url, headers=headers), timeout=60) as response:
                if response.status == 206:
                    match = re.fullmatch(r"bytes (\d+)-(\d+)/(\d+)", response.headers.get("Content-Range", ""))
                    if not match or tuple(map(int, match.groups())) != (offset, expected_size - 1, expected_size):
                        raise RuntimeError("Server returned an unexpected byte range; partial file preserved.")
                elif response.status == 200:
                    if offset:
                        print("Server did not support resume; restarting the partial download.", flush=True)
                    offset = 0
                else:
                    raise RuntimeError(f"Unexpected HTTP status: {response.status}")
                length = response.headers.get("Content-Length")
                if length is not None and int(length) != expected_size - offset:
                    raise RuntimeError("Remote archive size changed; check official metadata before updating the pinned checksum.")
                last_report = time.monotonic()
                with partial.open("ab" if offset else "wb") as output:
                    while chunk := response.read(CHUNK):
                        if offset + len(chunk) > expected_size:
                            raise RuntimeError("Response exceeds the expected archive size.")
                        output.write(chunk)
                        offset += len(chunk)
                        if time.monotonic() - last_report >= 10:
                            print(f"Downloaded {offset / 2**20:.1f} / {expected_size / 2**20:.1f} MiB ({offset / expected_size:.1%})", flush=True)
                            last_report = time.monotonic()
            if offset != expected_size:
                raise OSError(f"Connection ended at {offset} of {expected_size} bytes.")
            break
        except (URLError, OSError, TimeoutError, IncompleteRead) as exc:
            print(f"Transfer interrupted: {exc}. Partial download retained.", flush=True)
            if attempt == 5:
                raise RuntimeError("Download failed after five attempts. Rerun to resume the saved partial archive.") from exc
            time.sleep(min(attempt * 2, 10))
    verify_archive(partial, expected_size, expected_sha256)
    partial.rename(archive)
    print("Archive downloaded and checksum verified.", flush=True)


def safe_members(bundle):
    members = []
    seen = set()
    for info in bundle.infolist():
        name = info.filename
        path = PurePosixPath(name)
        parts = path.parts
        if (not parts or path.is_absolute() or "\\" in name
                or any(p in ("..", ".") or ":" in p or p.endswith((" ", "."))
                       or re.match(r"^(CON|PRN|AUX|NUL|COM[1-9]|LPT[1-9])(?:\.|$)", p, re.I)
                       for p in parts)
                or stat.S_ISLNK(info.external_attr >> 16)):
            raise RuntimeError(f"Unsafe ZIP member: {name!r}")
        key = str(path).casefold()
        if key in seen:
            raise RuntimeError(f"Duplicate ZIP member: {name!r}")
        seen.add(key)
        members.append((info, Path(*parts)))
    if not any(not info.is_dir() for info, _ in members):
        raise RuntimeError("Archive contains no dataset files.")
    return members


def checked_target(root, relative):
    target = root / relative
    if root.is_symlink() or target.is_symlink() or not target.resolve().is_relative_to(root.resolve()):
        raise RuntimeError(f"Extraction path escapes the dataset folder: {target}")
    return target


def matches_member(path, info):
    if not path.is_file() or path.stat().st_size != info.file_size:
        return False
    crc = 0
    with path.open("rb") as handle:
        while chunk := handle.read(CHUNK):
            crc = zlib.crc32(chunk, crc)
    return crc == info.CRC


def extract_archive(archive, destination):
    staging = destination.with_name("." + destination.name + ".extracting")
    existing = destination.exists()
    root = destination if existing else staging
    if root.is_symlink():
        raise RuntimeError(f"Dataset folder must not be a symlink: {root}")
    root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        members = safe_members(bundle)
        files = [(info, relative) for info, relative in members if not info.is_dir()]
        total_bytes = sum(info.file_size for info, _ in files)
        print(f"Checking {len(files):,} files; uncompressed size {total_bytes / 2**30:.2f} GiB.", flush=True)
        pending = []
        last_report = time.monotonic()
        for number, (info, relative) in enumerate(files, 1):
            target = checked_target(root, relative)
            if target.exists():
                if not matches_member(target, info):
                    raise RuntimeError(f"Existing dataset file is damaged or modified: {target}. Preserved; move it aside, then rerun.")
            elif existing:
                raise RuntimeError(f"Dataset file is missing: {target}. Move the incomplete dataset folder aside, then rerun to extract from the cached archive.")
            else:
                pending.append((info, relative))
            if time.monotonic() - last_report >= 10:
                print(f"Checked {number:,}/{len(files):,} files.", flush=True)
                last_report = time.monotonic()
        require_space(root, sum(info.file_size for info, _ in pending))
        for number, (info, relative) in enumerate(pending, 1):
            target = checked_target(root, relative)
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_name(target.name + ".osma-part")
            checked_target(root, temporary.relative_to(root))
            # A file is published only after ZipExtFile verifies its CRC at EOF.
            with bundle.open(info) as source, temporary.open("wb") as output:
                shutil.copyfileobj(source, output, CHUNK)
            temporary.replace(target)
            if number % 1000 == 0 or number == len(pending):
                print(f"Extracted {number:,}/{len(pending):,} files.", flush=True)
        counts = {
            "rgb": sum(bool(re.fullmatch(r"frame\d+\.jpg", p.name, re.I)) for _, p in files),
            "depth": sum(bool(re.fullmatch(r"depth\d+\.png", p.name, re.I)) for _, p in files),
            "semantic": sum(bool(re.fullmatch(r"semantic\d+\.png", p.name, re.I)) for _, p in files),
            "trajectories": sum(p.name == "traj.txt" for _, p in files),
            "validated_qa": sum(p.name.endswith("validated_questions.json") for _, p in files),
        }
        if not existing:
            staging.rename(destination)
        print("Verified existing extraction; no files extracted." if existing else "Extraction complete; all archive files passed ZIP CRC checks.", flush=True)
        return {"files": len(files), "uncompressed_bytes": total_bytes, "components": counts}


def main(project_root):
    project_root = Path(project_root)
    cache = project_root / "dataset" / "_downloads" / "osma-bench"
    destination = project_root / "dataset" / "osma-bench"
    cache.mkdir(parents=True, exist_ok=True)
    with download_lock(cache / ".download.lock"):
        archive = cache / "data.zip"
        download_archive(URL, archive, ARCHIVE_SIZE, ARCHIVE_SHA256)
        summary = extract_archive(archive, destination)
        record = {"repository": REPO_ID, "revision": REVISION, "url": URL,
                  "archive_sha256": ARCHIVE_SHA256, "archive_bytes": ARCHIVE_SIZE,
                  "destination": str(destination.resolve()), **summary}
        manifest = cache / "verified.json"
        temporary = manifest.with_suffix(".tmp")
        temporary.write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
        temporary.replace(manifest)
        print(json.dumps(summary, indent=2), flush=True)
        print(f"OSMa-Bench ready: {destination.resolve()}", flush=True)
        return destination


# Run the full download and verification in one cell.
cwd = Path.cwd().resolve()
project_root = next(
    (p for p in (cwd, *cwd.parents)
     if (p / "code" / "download_dependencies.ipynb").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Run from the capstone project folder or its code folder.")
OSMA_ROOT = main(project_root)
